# Day 1 — Introduction & URSim Setup

Welcome to Day 1 of the **UR5 AI Workshop**!

### 🎯 Learning Objectives
By the end of this notebook, you will:
1. Understand the UR5 robot’s architecture (joints, frames, communication).
2. Set up and run the URSim virtual robot.
3. Connect to URSim via Python using `ur_rtde`.
4. Send simple motion commands and read robot state data.


## 1. URSim Overview
URSim is a simulation environment provided by Universal Robots. It replicates the behavior of UR5/UR10 robots.

**Key features:**
- Test URScript programs without a physical robot.
- Network communication identical to real hardware.
- Useful for AI and control development.

**Common ports:**
- 30001–30004: URScript / dashboard communication.
- 30002: secondary client interface.
- 30004: RTDE interface (Real-Time Data Exchange).


## 🧩 Exercise 1 — URSim Installation and Configuration
**Goal:** Install URSim and verify that it runs locally.

**Steps:**
1. Install **URSim** (either standalone or in VirtualBox).
2. Configure network (use host-only adapter if in VirtualBox).
3. Launch URSim → *Polyscope GUI should appear*.
4. Set the robot to *Remote Control mode*.

**Deliverable:** Screenshot showing URSim running successfully.

## 🧩 Exercise 2 — Connect to URSim via Python

In [1]:
# Student Task
from rtde_control import RTDEControlInterface
from rtde_receive import RTDEReceiveInterface

ROBOT_IP = '127.0.0.1'  # Replace if needed
rtde_c = RTDEControlInterface(ROBOT_IP)
rtde_r = RTDEReceiveInterface(ROBOT_IP)

joint_positions = rtde_r.getActualQ()
print('Current joint positions:', joint_positions)

Current joint positions: [-1.5700000000000003, -1.5700000000000003, 1.57, -1.5700000000000003, -1.5700000000000003, 0.0]


## 🧩 Exercise 3 — Command a Simple Movement

In [2]:
# Student Task
import time
target_q = [0, -1.57, 1.57, 0, 1.57, 0]
rtde_c.moveJ(target_q, 0.5, 0.5)
time.sleep(2)
print('Reached target position')

Reached target position


## 🧩 Exercise 4 — Safety and Error Handling

In [ ]:
# Student Task
import numpy as np

def safe_move(target_q, speed=0.3):
    if len(target_q) != 6:
        print('Refused: need 6 joint values')
        return False
    if any(abs(q) > 2 * np.pi for q in target_q):
        print('Refused: joint out of range')
        return False
    if rtde_r.isProtectiveStopped() or rtde_r.isEmergencyStopped():
        print('Refused: robot is stopped')
        return False
    if not rtde_c.isJointsWithinSafetyLimits(target_q):
        print('Refused: outside safety limits')
        return False
    speed = min(speed, 1.0)  # never too fast
    rtde_c.moveJ(target_q, speed, speed)
    print('Moved to', target_q)
    return True

safe_move([0, -1.57, 1.57, 0, 1.57, 0])      # ok
safe_move([0, -1.57, 1.57, 0, 1.57])         # refused: 5 values
safe_move([7, -1.57, 1.57, 0, 1.57, 0])      # refused: 7 rad
safe_move([0, -1.57, 1.57, 0, 1.57, 0], 5)   # ok, speed limited to 1.0


## 🎓 Challenge — Log and Plot Robot Motion
**Goal:** Log the robot’s TCP position while moving and visualize the trajectory.

Hints:
- Use `getActualTCPPose()` inside a loop.
- Store data in a list, then plot with matplotlib.
- Save to `trajectory.csv` for later analysis.

In [ ]:
# Student Task
%matplotlib module://ipympl.backend_nbagg
import csv
import time
import matplotlib.pyplot as plt

rtde_c.moveJ([-1.57, -1.57, 1.57, -1.57, -1.57, 0], 0.5, 0.5)  # start pose
rtde_c.moveJ([0, -1.57, 1.57, 0, 1.57, 0], 0.5, 0.5, True)     # True = don't wait, so we can log while it moves

log = []
t0 = time.time()
while time.time() - t0 < 10:  # the move takes about 8 s
    log.append([time.time() - t0] + rtde_r.getActualTCPPose())
    time.sleep(0.02)  # 50 Hz

with open('trajectory.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['t', 'x', 'y', 'z', 'rx', 'ry', 'rz'])
    writer.writerows(log)

data = np.array(log)
ax = plt.figure().add_subplot(projection='3d')  # drag to rotate
ax.plot(data[:, 1], data[:, 2], data[:, 3])
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_zlabel('z (m)')
ax.set_title('TCP trajectory')
plt.show()


In [ ]:
rtde_c.stopScript()
rtde_c.disconnect()
rtde_r.disconnect()
